# Why You Need Polar Decomposition: Fast Differentiable SVD on GPU

This notebook demonstrates the accuracy of the **CANS-SVD** algorithm compared to existing SVD implementations in CUDA.

### 1. Clone the repository

In [ ]:
!git clone https://github.com/anonymous-502475/cans_svd.git

### 2. Install Dependencies

In [ ]:
!pip install -r cans_svd/requirements.txt

In [ ]:
%cd cans_svd

### 3. Import JAX and CANS-SVD

In [30]:
import jax
import jax.numpy as jnp
from jax import config
config.update("jax_default_matmul_precision", "float32")  # set default matmul precision to float32
assert jax.device_count('gpu') > 0, "GPU not available. Please use environment with GPU"

import tqdm
import scipy as sp
import pandas as pd
pd.options.display.float_format = '{:.1e}'.format

from src import cans_svd

In [5]:
def generate_matrix(size: int, cond_number: float, key) -> jnp.array:
    """
    Generate a random matrix with a specified condition number.

    Args:
        size (int): The size of the matrix (size x size).
        cond_number (float): The desired condition number of the matrix.

    Returns:
        jnp.array: The generated matrix.
    """
    key1, key2 = jax.random.split(key)

    u = jax.random.normal(key=key1, shape=(size, size), dtype=jnp.float32)
    u, _ = jax.lax.linalg.qr(u)

    v = jax.random.normal(key=key2, shape=(size, size), dtype=jnp.float32)
    v, _ = jax.lax.linalg.qr(v)

    s = jnp.logspace(0, jnp.log10(cond_number), size, base=10, dtype=jnp.float32)

    return u @ jnp.diag(s) @ v

In [6]:
def reconstruction_err(A: jnp.array, U: jnp.array, S: jnp.array, VT: jnp.array) -> float:
    """
    Compute the reconstruction error of the SVD decomposition.

    Args:
        A (jnp.array): The original matrix.
        U (jnp.array): The left singular vectors.
        S (jnp.array): The singular values.
        VT (jnp.array): The right singular vectors (transposed).

    Returns:
        float: The relative Frobenius norm of the reconstruction error.
    """
    A_reconstructed = U @ jnp.diag(S) @ VT

    return jnp.linalg.norm(A - A_reconstructed, ord='fro') / jnp.linalg.norm(A, ord='fro')

### 4. Check the accuracy of algorithms on an ill-conditioned matrix

First, lets generate random matrix:

In [32]:
# You can specify the matrix size and condition number here
SIZE = 4096                     # matrix size
N_TRIES = 1                     # number of tries for each method (increase for 100 to fully reproduce paper result)
CONDITION_NUMBERS = [           # condition number of the matrix
    10, 100, 1e4, 1e6
]

Now we can compare with different algorithms:
- Polar-based `jax.lax.linalg.SvdAlgorithm.POLAR`
- QR-based `jax.lax.linalg.SvdAlgorithm.QR`
- Jacobi-based `jax.lax.linalg.SvdAlgorithm.JACOBI`

In [33]:
data = []

for cond_number in CONDITION_NUMBERS:
    for try_idx in tqdm.tqdm(range(N_TRIES), desc=f"Condition number: {cond_number}"):
        A = generate_matrix(size=SIZE, cond_number=cond_number, key=jax.random.PRNGKey(try_idx))

        U_cans, S_cans, VT_cans = cans_svd(A)
        U_polar, S_polar, VT_polar = jax.lax.linalg.svd(A, algorithm=jax.lax.linalg.SvdAlgorithm.POLAR)
        U_qr, S_qr, VT_qr = jax.lax.linalg.svd(A, algorithm=jax.lax.linalg.SvdAlgorithm.QR)
        U_jacobi, S_jacobi, VT_jacobi = jax.lax.linalg.svd(A, algorithm=jax.lax.linalg.SvdAlgorithm.JACOBI)
        U_sp, S_sp, VT_sp = sp.linalg.svd(A)


        data.append({"cond": cond_number,
            "algorithm": "CANS SVD",
            "recon_err": reconstruction_err(A, U_cans, S_cans, VT_cans)
        })
        data.append({"cond": cond_number,
            "algorithm": "CUDA POLAR",
            "recon_err": reconstruction_err(A, U_polar, S_polar, VT_polar)
        })
        data.append({"cond": cond_number,
            "algorithm": "CUDA QR",
            "recon_err": reconstruction_err(A, U_qr, S_qr, VT_qr)
        })
        data.append({"cond": cond_number,
            "algorithm": "CUDA JACOBI",
            "recon_err": reconstruction_err(A, U_jacobi, S_jacobi, VT_jacobi)
        })
        data.append({"cond": cond_number,
            "algorithm": "SciPy",
            "recon_err": reconstruction_err(A, U_sp, S_sp, VT_sp)
        })

    df = pd.DataFrame(data)

Condition number: 1000000.0: 100%|██████████| 1/1 [00:48<00:00, 48.63s/it]


Let see relative reconstruction error:

In [34]:
df.pivot_table(index="algorithm", columns="cond", values="recon_err", aggfunc="mean")

cond,1.0e+01,1.0e+02,1.0e+04,1.0e+06
algorithm,,,,
CANS SVD,5.0e-06,4.4e-06,4.6e-06,5.8e-06
CUDA JACOBI,1.0e-03,8.2e-04,5.4e-04,5.7e-04
CUDA POLAR,4.2e-06,5.4e-06,7.6e+01,7.6e+01
CUDA QR,1.7e-05,2.6e-05,1.6e-05,1.2e-05
SciPy,3.2e-06,2.6e-06,2.6e-06,4.8e-06


As shown above Polar-based SVD has large reconstruction error on ill-conditioned matrix.

⚠️ **Important note:** Runtime experiments in the blog post were conducted on an NVIDIA B200 GPU, results may differ on older GPUs.